In [17]:
import os
import json
import re


# -------- CONFIG --------
IMAGE_DIR = "/Users/petitmi/petitmiweb/website-st/frontend/sunglasses/public/images/sunglasses"
DESC_FILE = "descriptions.txt"
OUTPUT_FILE = "products.json"
PRICE = "Upon Request"
MOQ = 20
# ------------------------


def to_camel_case(text):
    text = text.replace("&", "and")
    parts = re.split(r"[^\w]+", text.lower())
    return parts[0] + "".join(p.capitalize() for p in parts[1:])


def parse_descriptions(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        text = f.read()

    blocks = re.split(r"=+\s*(\d+(?:_\d+)?)\s*=+", text)

    results = {}

    for i in range(1, len(blocks), 2):
        key = blocks[i].strip()
        content = blocks[i + 1]

        name = ""
        colorway = ""
        description = ""
        sku = ""
        sections = {}
        current_section = None

        lines = content.splitlines()

        for raw in lines:
            line = raw.strip()

            if not line:
                continue

            # Meta fields
            if line.startswith("Name:"):
                name = line.replace("Name:", "").strip()
                continue

            if line.startswith("Colorway:"):
                colorway = line.replace("Colorway:", "").strip()
                continue

            if line.startswith("Description:"):
                description = line.replace("Description:", "").strip()
                continue

            # Section
            if line.startswith("###"):
                section_name = line.replace("###", "").strip()
                section_key = to_camel_case(section_name)
                current_section = section_key
                sections[current_section] = {}
                continue

            # Key: Value
            if ":" in line and current_section:
                k, v = line.split(":", 1)
                key_name = to_camel_case(k.strip())

                sections[current_section][key_name] = v.strip()

        results[key] = {
            "name": name,
            "colorway": colorway,
            "description": description,
            "details": sections,
        }

    return results


def scan_images(folder):
    images = {}

    for f in os.listdir(folder):
        if not f.endswith(".png"):
            continue

        name = f.replace(".png", "")
        parts = name.split("_")

        if len(parts) == 3:
            model, variant, view = parts
            key = f"{model}_{variant}"

        elif len(parts) == 2:
            model, view = parts
            key = model

        else:
            continue

        if key not in images:
            images[key] = {}

        images[key][view] = f"/store/images/sunglasses/{f}"

    return images


def build_products(descs, images):
    products = {}
    index = 1

    grouped = {}

    for key in images:

        if "_" in key:
            model, variant = key.split("_")
        else:
            model = key
            variant = "0"

        model_id = int(model)
        variety = int(variant)
        sku = f"{model}-{variant}"

        meta = descs.get(key, {})

        variant_obj = {
            "variety": variety,
            "sku": sku,
            "colorway": meta.get("colorway"),
            "imageFront": images[key].get("front"),
            "imageSide": images[key].get("side"),
            "details": meta.get("details", {}),
            
        }

        if model_id not in grouped:
            grouped[model_id] = {
                "id": model_id,
                "model": model_id,
                "name": meta.get("name"),
                "description": meta.get("description"),
                "price": PRICE,
                "moq": MOQ,
                "variants": []
            }

        grouped[model_id]["variants"].append(variant_obj)

    for model_id, product in grouped.items():
        products[index] = product
        index += 1

    return products


def main():

    print("📦 Parsing descriptions...")
    descs = parse_descriptions(DESC_FILE)

    print("🖼️  Scanning images...")
    images = scan_images(IMAGE_DIR)

    print("🔧 Building products...")
    products = build_products(descs, images)

    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(products, f, indent=2, ensure_ascii=False)

    print(f"✅ Done! Saved to {OUTPUT_FILE}")


if __name__ == "__main__":
    main()


📦 Parsing descriptions...
🖼️  Scanning images...
🔧 Building products...
✅ Done! Saved to products.json
